# Практика 9. Перетворення даних

**Студент:** Yurii_huz  
**Група:** IT-41  
**Варіант:** 1, Київ  
**Базова ціна:** 850 грн  
**Базова кількість:** 3

Мета: створити похідні змінні, дискретизувати вік, масштабувати числові ознаки та закодувати місто доставки.

In [ ]:
import numpy as np
import pandas as pd

base_price = 850
base_quantity = 3
customers = pd.DataFrame({
    "вік": [19, 24, 31, 38, 45, 52, 58, 63, 27, 41],
    "дохід": [18000, 24500, 32000, 41000, 53000, 67000, 82000, 96000, 28500, 46000],
    "кількість": [2, 4, 3, 6, 1, 5, 3, 2, 4, 7],
    "місто_доставки": ["Київ", "Львів", "Одеса", "Харків", "Київ", "Дніпро", "Львів", "Одеса", "Київ", "Харків"],
})
print(customers)
print("Унікальні міста:", customers["місто_доставки"].unique())

## Завдання 2. Похідні змінні

`частка_доходу` показує, яку частину місячного доходу клієнт витрачає на замовлення. Це дає змогу порівнювати фінансове навантаження між клієнтами з різними доходами, тоді як сама сума замовлення цього не показує.

In [ ]:
customers["сума_замовлення"] = base_price * customers["кількість"]
customers["частка_доходу"] = customers["сума_замовлення"] / customers["дохід"]
print(customers[["дохід", "кількість", "сума_замовлення", "частка_доходу"]])

## Завдання 3. Дискретизація віку

`cut()` створює інтервали однакової довжини, а `qcut()` підбирає межі так, щоб у групах було приблизно однаково спостережень.

In [ ]:
customers["вікова_група_cut"] = pd.cut(customers["вік"], bins=3)
customers["вікова_група_qcut"] = pd.qcut(customers["вік"], q=3)
print("Кількість клієнтів у групах cut:")
print(customers["вікова_група_cut"].value_counts().sort_index())
print()
print("Кількість клієнтів у групах qcut:")
print(customers["вікова_група_qcut"].value_counts().sort_index())

Для `cut()` межі визначаються діапазоном віку, тому за нерівномірного розподілу клієнтів кількість у групах різниться. `qcut()` використовує квантілі й намагається розкласти однакову кількість рядків у кожну групу, тому його межі можуть бути нерівними.

## Завдання 4. Масштабування

In [ ]:
for column in ["вік", "дохід"]:
    minimum = customers[column].min()
    maximum = customers[column].max()
    customers[column + "_minmax"] = (customers[column] - minimum) / (maximum - minimum)
    customers[column + "_z"] = (customers[column] - customers[column].mean()) / customers[column].std()

print(customers[["вік_minmax", "дохід_minmax", "вік_z", "дохід_z"]].head())

До масштабування вік і дохід мають різні одиниці та порядок величини: дохід у тисячах гривень чисельно набагато більший за вік. Метод, чутливий до масштабу, міг би надати доходу непропорційно великий вплив. Min-max приводить значення до `[0, 1]`, а z-оцінка — до відхилень у стандартних відхиленнях від середнього.

## Завдання 5. Індикаторне кодування

In [ ]:
city_dummies = pd.get_dummies(customers["місто_доставки"], prefix="місто")
customers_encoded = pd.concat([customers, city_dummies], axis=1)
print(city_dummies)
print("Кількість міст:", customers["місто_доставки"].nunique())
print("Кількість dummy-стовпців:", city_dummies.shape[1])
print(customers_encoded.head())

## Письмові відповіді

1. `pd.cut()` ділить діапазон на інтервали однакової довжини, а `pd.qcut()` використовує квантилі та забезпечує приблизно однакову кількість спостережень у групах.

2. Місто є номінальною змінною без природного порядку, тому one-hot кодування створює окремий індикатор для кожної категорії. Числове кодування 1, 2, 3 помилково створило б штучну відстань і порядок між містами.

3. Масштабування потрібне перед дистанційними методами та регресією з регуляризацією, щоб змінна з великими числовими одиницями не домінувала над змінною з малим масштабом.

4. Частка доходу поєднує суму замовлення і дохід у відносний показник. Вона краще показує фінансове навантаження покупки та дає змогу порівнювати клієнтів із різними рівнями доходу.

## Підсумок

Для клієнтів інтернет-магазину Києва побудовано похідні змінні, виконано дискретизацію віку через `cut` і `qcut`, масштабування віку та доходу двома способами й one-hot кодування міст доставки.